# Session 04: Data Collection Using Pandas
## Part 1: Importing and Handling Different Data Formats

**Project:** Fictional Housing Society Electricity Readings

This notebook covers data generation, CSV/Excel/TSV/JSON imports, data quality detection and chunk-based processing.

In [ ]:
import numpy as np
import pandas as pd

print("NumPy Version:", np.__version__)
print("Pandas Version:", pd.__version__)

## Task 2: Generate 1,200 Monthly Electricity Readings

Generate readings for 50 flats across 12 months in 2025. Save the dataset in CSV, semicolon-separated CSV, TSV, Excel and JSON formats.

In [ ]:
np.random.seed(42)

# Generate 50 flat IDs: A-01 to D-25
flat_ids = [f"{letter}-{number:02d}" for letter in "ABCD" for number in range(1, 26)]

# Generate 12 months
months = pd.date_range("2025-01-01", periods=12, freq="MS").strftime("%Y-%m")

# Generate 1,200 readings
readings = pd.DataFrame({
    "flat_id": np.repeat(flat_ids, 12),
    "month": np.tile(months, 50),
    "units_kwh": np.random.randint(60, 421, size=1200)
})

readings["amount_inr"] = readings["units_kwh"] * 7.5
readings["paid"] = np.random.choice(["Yes", "No"], size=1200, p=[0.9, 0.1])

print("Generated DataFrame Shape:", readings.shape)
display(readings.head())

# Save in five formats
readings.to_csv("readings_comma.csv", index=False)
readings.to_csv("readings_semicolon.csv", sep=";", index=False)
readings.to_csv("readings_tab.tsv", sep="\t", index=False)
readings.to_excel("readings.xlsx", index=False)
readings.to_json("readings.json", orient="records")

print("\nAll five reading files saved successfully.")

## Task 1: Create Messy Data and Detect Data Quality Issues

Create readings_messy.csv by introducing 5% missing units_kwh values, text-based amount_inr values and inconsistent paid column casing. Detect the issues without fixing them.

In [ ]:
# Create a copy of the original readings
messy_readings = readings.copy()

# Make 5% of units_kwh values missing
missing_count = int(len(messy_readings) * 0.05)
missing_indices = np.random.choice(
    messy_readings.index,
    size=missing_count,
    replace=False
)
messy_readings.loc[missing_indices, "units_kwh"] = np.nan

# Convert amount_inr values into text
messy_readings["amount_inr"] = messy_readings["amount_inr"].map(
    lambda amount: f"Rs {amount:.2f}"
)

# Mix the case of paid values
messy_readings["paid"] = [
    value.lower() if index % 3 == 0
    else value.upper() if index % 3 == 1
    else value
    for index, value in enumerate(messy_readings["paid"])
]

# Save messy dataset
messy_readings.to_csv("readings_messy.csv", index=False)

# Load messy dataset
df = pd.read_csv("readings_messy.csv")

print("DataFrame Information:")
df.info()

print("\nDataFrame Description:")
display(df.describe(include="all"))

# Detect issues without fixing them
issue_report = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[column].dtype) for column in df.columns],
    "missing_count": df.isnull().sum().values,
    "unique_count": df.nunique(dropna=True).values
})

issue_report.to_csv("issue_report.csv", index=False)

print("\nIssue Report:")
display(issue_report)

print("\nMessy data and issue report saved successfully.")

## Task 3: Load and Compare All Five File Formats

Load the five generated files using the correct Pandas functions and separators. Compare each file's shape and total units_kwh.

In [ ]:
# Load all five file formats
df_comma = pd.read_csv("readings_comma.csv")
df_semicolon = pd.read_csv("readings_semicolon.csv", sep=";")
df_tab = pd.read_csv("readings_tab.tsv", sep="\t")
df_excel = pd.read_excel("readings.xlsx")
df_json = pd.read_json("readings.json")

# Store loaded DataFrames
loaded_files = {
    "readings_comma.csv": df_comma,
    "readings_semicolon.csv": df_semicolon,
    "readings_tab.tsv": df_tab,
    "readings.xlsx": df_excel,
    "readings.json": df_json
}

# Compare shape and total units
comparison = pd.DataFrame([
    {
        "file_name": file_name,
        "shape": str(data.shape),
        "units_kwh_sum": data["units_kwh"].sum()
    }
    for file_name, data in loaded_files.items()
])

print("File Comparison:")
display(comparison)

# Verify all five totals match
all_totals = [data["units_kwh"].sum() for data in loaded_files.values()]
all_match = all(np.isclose(total, all_totals[0]) for total in all_totals)

print("\nDo all five files have matching units_kwh totals?", all_match)

## Task 4: Process a Large CSV File Using Chunks

Generate 60,000 hourly meter readings and process the CSV in chunks of 10,000 rows. Compare chunk-based totals with the original DataFrame.

In [ ]:
# Generate 60,000 hourly meter readings
np.random.seed(42)

meter_data = pd.DataFrame({
    "flat_id": np.resize(flat_ids, 60000),
    "hour_no": np.arange(1, 60001),
    "kwh": np.round(np.random.uniform(0.1, 2.5, size=60000), 2)
})

# Save large dataset
meter_data.to_csv("meter_big.csv", index=False)

print("Generated DataFrame Shape:", meter_data.shape)
print("Original Total Rows:", len(meter_data))
print("Original Total kWh:", meter_data["kwh"].sum())

# Read the large CSV in chunks
chunk_rows = 0
chunk_kwh_total = 0.0
chunk_count = 0

for chunk in pd.read_csv("meter_big.csv", chunksize=10000):
    chunk_rows += len(chunk)
    chunk_kwh_total += chunk["kwh"].sum()
    chunk_count += 1

# Compare chunk totals against original DataFrame
original_rows = len(meter_data)
original_kwh_total = meter_data["kwh"].sum()

print("\nChunk Count:", chunk_count)
print("Rows Processed:", chunk_rows)
print("Chunk-based Total kWh:", chunk_kwh_total)

print("\nRow Counts Match:", chunk_rows == original_rows)
print("kWh Totals Match:", np.isclose(chunk_kwh_total, original_kwh_total, atol=0.01))

## Final Summary

- Generated 1,200 electricity readings for a fictional housing society.
- Saved the readings in CSV, semicolon CSV, TSV, Excel and JSON formats.
- Created a messy dataset and detected missing values and inconsistent data types.
- Compared all five imported datasets.
- Processed 60,000 meter readings in six chunks of 10,000 rows.
- Verified row counts and kWh totals using NumPy and Pandas.